In [ ]:
# Imports
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
import lightgbm as lgb
import joblib


# 1. Charger le dataset et filtrer TCP

df = pd.read_csv("final_dataset.csv")
print("Shape before filtering:", df.shape)

df = df[df['Protocol'] == 'TCP'].copy()
print("Shape after TCP filter:", df.shape)
print(df['Protocol'].value_counts())

# 2. Créer la colonne 'loss' depuis Info
df['loss'] = df['Info'].apply(
    lambda x: 1 if isinstance(x, str) and ('Retransmission' in x or 'Dup ACK' in x) else 0
)

# 3. Imputation des NaN
# RTT par médiane par niveau de congestion
df['RTT'] = df.groupby('congestion_level')['RTT'].transform(
    lambda x: x.fillna(x.median())
)

# Length par médiane globale
df['Length'] = df['Length'].fillna(df['Length'].median())

# Si d'autres colonnes ont des NaN, les remplir par 0
for col in ['delta_time', 'rtt_std_5', 'loss_burst_5']:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# 4. Feature engineering
df = df.sort_values('Time')

# delta_time
df['delta_time'] = df['Time'].diff().fillna(0)

# RTT rolling std sur 5 paquets
df['rtt_std_5'] = df['RTT'].rolling(5).std().fillna(0)

# pertes groupées sur 5 paquets
df['loss_burst_5'] = df['loss'].rolling(5).sum().fillna(0)

# 5. Retirer les outliers
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return df[(df[column] >= lower) & (df[column] <= upper)]

for col in ['RTT', 'Length', 'delta_time']:
    df = remove_outliers(df, col)

# 6. Préparer X et y

features = ['RTT',
            'Length',
            'loss',
            'delta_time',
            'rtt_std_5',
            'loss_burst_5']

X = df[features]
y = df['congestion_level']

# Standardisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# 7. Entraînement LightGBM
lgbm = lgb.LGBMClassifier(
    n_estimators=400, #400 trees
    learning_rate=0.05,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

lgbm.fit(X_train, y_train)

# 8. Évaluation
y_pred = lgbm.predict(X_test)

print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# 9. Sauvegarder le modèle
joblib.dump(lgbm, "lgbm_congestion_model.pkl")
joblib.dump(scaler, "scaler.pkl")
print("Modèle et scaler sauvegardés !")


Shape before filtering: (412665, 9)
Shape after TCP filter: (324747, 9)
Protocol
TCP    324747
Name: count, dtype: int64


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Balanced Accuracy: 0.8830790427993943
Confusion Matrix:
 [[26150   804   853]
 [  377 17925  2807]
 [   47   136  1121]]

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.94      0.96     27807
           1       0.95      0.85      0.90     21109
           2       0.23      0.86      0.37      1304

    accuracy                           0.90     50220
   macro avg       0.72      0.88      0.74     50220
weighted avg       0.95      0.90      0.92     50220

Modèle et scaler sauvegardés !
